<a href="https://colab.research.google.com/github/Liex2810/FAKTAalone/blob/main/02_eda_augmenting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔤 EDA (Easy Data Augmentation)

Notebook ini melakukan **Easy Data Augmentation (EDA)** untuk menambah variasi data training sebelum proses training LSTM.

Pipeline:

```
Clean Dataset
      ↓
EDA
      ↓
Augmented Training Dataset
```


In [14]:
# Import Library
import random
import pandas as pd
from typing import List

print("Libraries loaded.")

Libraries loaded.


In [15]:
# Load Dataset
# Ganti path sesuai lokasi dataset hasil processing
DATA_PATH = "dataset_hasil_processing (3).csv"

df = pd.read_csv(DATA_PATH)

print(df.head())
print(f"Total data: {len(df)}")

                                                text label  \
0  ternyata ini kodenya pantesan tetangga gak per...  hoax   
1  purbaya yudhi sadewa butuh dana bantuan tenang...  hoax   
2  memanas guys menkeu purbaya tantang luhut menk...  hoax   
3  megawati soekarnoputri ingatkan kalo mau lihat...  hoax   
4  silahkan hubungi whatsap untuk mendaftar 08584...  hoax   

                                          clean_text  \
0  ternyata ini kodenya pantesan tetangga gak per...   
1  purbaya yudhi sadewa butuh dana bantuan tenang...   
2  memanas guys menkeu purbaya tantang luhut menk...   
3  megawati soekarnoputri ingatkan kalo mau lihat...   
4  silahkan hubungi whatsap untuk mendaftar 08584...   

                                     normalized_text  word_count  char_count  
0  ternyata ini kodenya pantesan tetangga tidak p...          14          94  
1  purbaya yudhi sadewa butuh dana bantuan tenang...          44         313  
2  memanas guys menkeu purbaya tantang luhut menk... 

## Synonym Dictionary

Tambahkan atau ubah sinonim sesuai kebutuhan.

In [16]:
INDONESIAN_SYNONYMS = {
    # Provocative words
    "viral": ["heboh", "geger", "terbongkar"],
    "heboh": ["viral", "geger", "gempar"],
    "geger": ["viral", "heboh", "gempar"],
    "sebarkan": ["sebarluaskan", "bagikan", "sampaikan"],
    "sebarluaskan": ["sebarkan", "bagikan", "kirimkan"],
    "bahaya": ["berbahaya", "berisiko", "mengancam"],
    "berbahaya": ["bahaya", "berisiko", "mengancam"],
    "waspada": ["awas", "hati-hati", "siaga"],
    "awas": ["waspada", "hati-hati", "peringatan"],
    "terbongkar": ["terkuak", "ungkap", "viral"],
    "terkuak": ["terbongkar", "ungkap", "viral"],

    # Urgency words
    "segera": ["secepatnya", "segeralah", "sekarang"],
    "secepatnya": ["segera", "segeralah", "sekarang"],
    "penting": ["krusial", "urgen", "vital"],
    "krusial": ["penting", "urgen", "vital"],
    "darurat": ["genting", "kritis", "mendesak"],
    "genting": ["darurat", "kritis", "mendesak"],
    "mendesak": ["darurat", "penting", "krusial"],

    # Common verbs
    "mengatakan": ["menyatakan", "mengungkapkan", "mengklaim"],
    "menyatakan": ["mengatakan", "mengungkapkan", "mengklaim"],
    "mengungkapkan": ["mengatakan", "menyatakan", "memaparkan"],
    "mengklaim": ["mengatakan", "menyatakan", "menegaskan"],
    "menegaskan": ["mengklaim", "menyatakan", "menegaskan"],
    "menunjukkan": ["menampilkan", "memperlihatkan", "menampakkan"],
    "menampilkan": ["menunjukkan", "memperlihatkan", "menayangkan"],
    "memperlihatkan": ["menunjukkan", "menampilkan", "menampakkan"],
    "mendapatkan": ["memperoleh", "mendapat", "meraih"],
    "memperoleh": ["mendapatkan", "mendapat", "meraih"],
    "mendapat": ["mendapatkan", "memperoleh", "meraih"],
    "membuat": ["membikin", "menciptakan", "menghasilkan"],
    "membikin": ["membuat", "menciptakan", "menghasilkan"],
    "menciptakan": ["membuat", "membikin", "menghasilkan"],
    "melakukan": ["menjalankan", "melaksanakan", "mengerjakan"],
    "menjalankan": ["melakukan", "melaksanakan", "mengerjakan"],
    "melaksanakan": ["melakukan", "menjalankan", "mengerjakan"],
    "mengumumkan": ["menyatakan", "memberitahukan", "menginformasikan"],
    "memberitahukan": ["mengumumkan", "menyatakan", "menginformasikan"],

    # Adjectives
    "besar": ["akbar", "raksasa", "masif"],
    "akbar": ["besar", "raksasa", "masif"],
    "raksasa": ["besar", "akbar", "masif"],
    "kecil": ["mungil", "mini", "cilik"],
    "mungil": ["kecil", "mini", "cilik"],
    "banyak": ["berlimpah", "melimpah", "segudang"],
    "berlimpah": ["banyak", "melimpah", "segudang"],
    "cepat": ["lekas", "segera", "begas"],
    "lambat": ["pelan", "perlahan", "perlahan-lahan"],
    "parah": ["mengerikan", "fatal", "gawat"],
    "mengerikan": ["parah", "fatal", "mencekam"],
    "fatal": ["parah", "mengerikan", "berbahaya"],
    "resmi": ["sah", "legal", "formal"],
    "sah": ["resmi", "legal", "formal"],

    # Nouns
    "orang": ["individu", "seseorang", "pihak"],
    "individu": ["orang", "seseorang", "pihak"],
    "tempat": ["lokasi", "area", "wilayah"],
    "lokasi": ["tempat", "area", "wilayah"],
    "wilayah": ["tempat", "lokasi", "area"],
    "waktu": ["masa", "saat", "ketika"],
    "masalah": ["persoalan", "permasalahan", "kasus"],
    "persoalan": ["masalah", "permasalahan", "kasus"],
    "solusi": ["pemecahan", "jawaban", "jalan keluar"],
    "pemerintah": ["pemerintahan", "negara", "rezim"],
    "masyarakat": ["warga", "publik", "rakyat"],
    "warga": ["masyarakat", "publik", "rakyat"],
    "rakyat": ["masyarakat", "warga", "publik"],
}

In [17]:
def get_synonyms(word:str)->List[str]:
    return INDONESIAN_SYNONYMS.get(word.lower(),[])

def synonym_replacement(words,n):
    words=words.copy()
    idx=[i for i,w in enumerate(words) if get_synonyms(w)]
    if not idx:
        return words
    for i in random.sample(idx,min(n,len(idx))):
        words[i]=random.choice(get_synonyms(words[i]))
    return words

def random_insertion(words,n):
    words=words.copy()
    for _ in range(n):
        candidates=[w for w in words if get_synonyms(w)]
        if not candidates:
            break
        w=random.choice(candidates)
        words.insert(random.randint(0,len(words)),random.choice(get_synonyms(w)))
    return words

def random_swap(words,n):
    words=words.copy()
    for _ in range(min(n,len(words)//2)):
        i,j=random.sample(range(len(words)),2)
        words[i],words[j]=words[j],words[i]
    return words

def random_deletion(words,p):
    if len(words)<=1:
        return words
    new=[w for w in words if random.random()>p]
    return new if new else [random.choice(words)]

def eda_augment(text,alpha_sr=0.1,alpha_ri=0.1,alpha_rs=0.1,p_rd=0.1,num_aug=2):
    words=text.split()
    if len(words)<2:
        return [text]*num_aug

    n_sr=max(1,int(alpha_sr*len(words)))
    n_ri=max(1,int(alpha_ri*len(words)))
    n_rs=max(1,int(alpha_rs*len(words)))

    result=[]
    for _ in range(num_aug):
        aug=words.copy()
        aug=synonym_replacement(aug,n_sr)
        aug=random_insertion(aug,n_ri)
        aug=random_swap(aug,n_rs)
        aug=random_deletion(aug,p_rd)
        result.append(" ".join(aug))
    return result

print("EDA Functions Loaded.")

EDA Functions Loaded.


In [18]:
# Demo EDA

sample=df.iloc[0]["text"]

print("Original:")
print(sample)

print("\nAugmented:")
for i,t in enumerate(eda_augment(sample,num_aug=3),1):
    print(f"{i}. {t}")

Original:
ternyata ini kodenya pantesan tetangga gak pernah isi token gratis 3 bulan pemakaian listrik

Augmented:
1. ternyata ini kodenya pantesan tetangga isi pernah gak token gratis 3 bulan pemakaian listrik
2. ternyata token kodenya pantesan tetangga gak pernah ini gratis bulan pemakaian
3. tetangga ini kodenya pantesan ternyata gak pernah isi token gratis 3 bulan pemakaian listrik


In [19]:
# Apply EDA ke seluruh dataset

augmented=[]

for _,row in df.iterrows():
    augmented.append({
        "text":row["text"],
        "label":row["label"]
    })

    for text in eda_augment(str(row["text"]),num_aug=2):
        augmented.append({
            "text":text,
            "label":row["label"]
        })

aug_df=pd.DataFrame(augmented)

print(aug_df.head())
print(f"Jumlah data setelah EDA: {len(aug_df)}")

                                                text label
0  ternyata ini kodenya pantesan tetangga gak per...  hoax
1  ternyata isi kodenya pantesan tetangga gak per...  hoax
2  ternyata ini kodenya pantesan token gak pernah...  hoax
3  purbaya yudhi sadewa butuh dana bantuan tenang...  hoax
4  purbaya umkm sadewa butuh dana tenang, dana ar...  hoax
Jumlah data setelah EDA: 23214


In [20]:
import os

OUTPUT_PATH = "data/processed/train_augmented.csv"

# Buat folder jika belum ada
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Simpan hasil
aug_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to: {OUTPUT_PATH}")

Saved to: data/processed/train_augmented.csv


## Output

Notebook ini menghasilkan:

- `train_augmented.csv`

File tersebut digunakan pada notebook **03_LSTM_Architecture.ipynb**.